# CK-M worked solution: the empirical variant (Prairie Town's peaks)

*Energy Systems, Step by Step* · Modeler checkpoint CK-M. The design, the code, the answers to tasks 2 to 5 and 9 to 10, and the check against EIA's published 2025 capacity factor. Put `town_year.json` (from the checkpoint page) next to this notebook first.

**Runs with** Python 3.9 or later (for `zoneinfo`), NumPy and SciPy 1.9 or later. On Windows, `zoneinfo` needs time-zone data (`pip install tzdata`); without it the notebook uses Central time's 2025 clock rule, written out.

In [ ]:
import json, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"
import datetime as dt
try:
    from zoneinfo import ZoneInfo
    CENTRAL = ZoneInfo("America/Chicago")
except Exception:                     # no time-zone data: Central time in 2025, daylight time 9 March to 2 November
    CENTRAL = None

## The design, before computing

- **Question:** descriptive. How much do the wind and solar farms give in the hours the town needs power most, against their year averages?
- **Metrics:** average output in each set of hours, as % of the farm's size and in MW.
- **Scaling:** demand × 450 MW, wind × 150 MW, solar × 200 MW. So 100% means SPP's highest 2025 hour, not installed capacity: the year's mean output in % is not a capacity factor.
- **Hours:** the 100 highest by demand, and by net load (demand minus wind and solar); the earlier hour first where values tie.
- **Left out:** one weather year, outages, imports, an adequacy model. An average over top hours is a screen, not a capacity credit.

In [ ]:
year = json.load(open("town_year.json"))
d = np.array(year["demand"]) / 1000 * 450
w = np.array(year["wind"]) / 1000 * 150
s = np.array(year["solar"]) / 1000 * 200
net = d - w - s
# checks before any analysis
assert len(d) == len(w) == len(s) == 8760
assert d.max() == 450 and w.max() == 150 and s.max() == 200 and min(d.min(), w.min(), s.min()) >= 0
print(f"{len(d):,} hours, {d.sum() / 1000:,.1f} GWh of demand; the file's time stamps are hour-ending from {year['start_utc']} UTC")

In [ ]:
def top(x, n=100):
    return np.argsort(-x, kind="stable")[:n]        # highest first; a stable sort keeps the earlier hour first on ties

tD, tN = top(d), top(net)
cf_w, cf_s = w.mean() / 150 * 100, s.mean() / 200 * 100
rows = {"whole year": (cf_w, cf_s, w.mean() + s.mean()),
        "100 highest demand": (w[tD].mean() / 1.5, s[tD].mean() / 2, (w[tD] + s[tD]).mean()),
        "100 highest net load": (w[tN].mean() / 1.5, s[tN].mean() / 2, (w[tN] + s[tN]).mean())}
for k, (a, b, c) in rows.items():
    print(f"{k:22s} wind {a:5.2f}%  solar {b:5.2f}%  both {c:6.2f} MW")
assert abs(cf_w - 52.75) < 0.01 and abs(rows["100 highest demand"][0] - 58.11) < 0.01 and abs(rows["100 highest demand"][1] - 68.97) < 0.01
assert abs(rows["100 highest net load"][2] - 52.89) < 0.01

**Where the hours fall.** The file's index is the hour ending at `start_utc` plus *i* hours, in UTC; convert to Central time to read the hours.

In [ ]:
t0 = dt.datetime.fromisoformat(year["start_utc"]).replace(tzinfo=dt.timezone.utc)
def to_central(u):
    if CENTRAL is not None:
        return u.astimezone(CENTRAL)
    daylight = dt.datetime(2025, 3, 9, 8, tzinfo=dt.timezone.utc) <= u < dt.datetime(2025, 11, 2, 7, tzinfo=dt.timezone.utc)
    return (u - dt.timedelta(hours=5 if daylight else 6)).replace(tzinfo=None)
begin = lambda i: to_central(t0 + dt.timedelta(hours=int(i) - 1))   # hour beginning, local
for name, idx in [("demand", tD), ("net load", tN)]:
    months = sorted({begin(i).month for i in idx}); hours = sorted({begin(i).hour for i in idx})
    print(f"100 highest {name}: months {months}, local hours beginning {hours}")

## Validation against a published benchmark (tasks 9 to 11)

EIA's *Electric Power Monthly*, Table 6.07.B, gives 33.8% for the U.S. utility-scale wind fleet in 2025 and 24.0% for solar. The dataset's wind figure is far above it, but it is a different quantity: its shapes are shares of their own highest hour, not of capacity, and SPP's fleet and wind differ from the U.S. average. The year file alone cannot split the gap, so this comparison checks a definition; it does not validate the number. The like-for-like check below adds SPP's wind capacity.

**Rescaling wind to 33.8%.** For the same 100 net-load hours, every ratio to the farm's own average survives. Picking the net-load hours again after rescaling is also valid, and changes both the hours and the ratio: say which you did.

In [ ]:
gap = cf_w - 33.8
k = 0.338 / (cf_w / 100)
w2 = w * k
ratio, ratio2 = w[tN].mean() / w.mean(), w2[tN].mean() / w2.mean()       # the same hours
tN2 = top(d - w2 - s)                                                    # the net-load hours picked again
ratio3, kept = w2[tN2].mean() / w2.mean(), len(set(tN2.tolist()) & set(tN.tolist()))
print(f"Wind: dataset {cf_w:.2f}% of its highest hour against EIA's 33.8% capacity factor: {gap:.2f} percentage points; solar {cf_s:.2f}% against 24.0%")
print(f"Net-load hours ÷ year average: {ratio:.4f} before rescaling, {ratio2:.4f} after (the same hours)")
print(f"Hours picked again after rescaling: {kept} of 100 stay; the ratio is {ratio3:.4f}")
assert abs(gap - 18.95) < 0.01 and abs(ratio - 0.5606) < 1e-4 and abs(ratio - ratio2) < 1e-12
assert abs(ratio3 - 0.7138) < 1e-4 and kept == 82

**A like-for-like check: a derived estimate.** SPP's 2025 wind output divided by its wind capacity as EIA counts it. The capacity is independent of the year file; the output is the same EIA-930 data, so this checks the reading of the gap, not the dataset's values.

- **Output:** 110.7 TWh in 2025: EIA-930 adjusted wind for the Southwest Power Pool, cleaned as the course's year file is.
- **Capacity:** 33,385 MW: EIA-860M (december_generator2025.xlsx, end of December 2025), onshore wind units in operation, time-adjusted net summer capacity by the rule of Table 6.07.B.
- **Limits:** one year; net summer capacity, not nameplate; the units EIA lists in SPP's area, which may not match the wind the balancing area meters exactly.

In [ ]:
spp_output_mwh = 110726210.4        # EIA-930, 2025, cleaned as the year file is (frozen hours replaced)
spp_wind_mw = 33384.7              # EIA-860M, time-adjusted net summer MW
cf_spp = spp_output_mwh / (spp_wind_mw * 8760) * 100
definition, fleet = cf_w - cf_spp, cf_spp - 33.8
print(f"SPP's 2025 wind capacity factor, like for like: {cf_spp:.1f}%")
print(f"Of the {gap:.1f}-point gap: {definition:.1f} points from the dataset's definition, {fleet:.1f} from SPP's fleet and wind against the U.S. average")
assert abs(cf_spp - 37.862) < 0.01 and abs(definition + fleet - gap) < 1e-9